In [ ]:


CODE_SOURCE = "kaggle_dataset"
CODE_DATASET_DIR = "/kaggle/input/onevoice-kaggle-code"
REPO_DIR = "/kaggle/working/onevoice"

DATASET_TT_DIR = "/kaggle/input/onevoice-dolphin-tier-a/tt"

REPO_URL = ""
REPO_BRANCH = "master"
USE_GITHUB_TOKEN = False

MODEL_ID = "JusperLee/Dolphin"
SEGMENT_SECONDS = 2.0
MAX_CLIPS = 8
WINDOW_MS = 2000.0
OUTPUT_BUFFER_MS = 100.0

EXPORT_TARGETS = [
    ("corpus_mix_02", "s2"),
    ("corpus_mix_03", "s1"),
]
EXPORT_ROOT = "/kaggle/working/separated_outputs"

print("CODE_SOURCE =", CODE_SOURCE)
print("CODE_DATASET_DIR =", CODE_DATASET_DIR)
print("DATASET_TT_DIR =", DATASET_TT_DIR)


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.environ["TORCHDYNAMO_DISABLE"] = "1"

repo = Path(REPO_DIR)
if repo.exists():
    print(f"Removing existing {repo}")
    shutil.rmtree(repo)
repo.mkdir(parents=True, exist_ok=True)

def _find_pyproject_root(start: Path) -> Path:
    if (start / "pyproject.toml").is_file():
        return start
    for hit in Path("/kaggle/input").rglob("pyproject.toml"):
        return hit.parent
    raise RuntimeError(
        f"pyproject.toml not found under {start} or /kaggle/input. "
        "Attach dataset onevoice-kaggle-code."
    )

if CODE_SOURCE == "kaggle_dataset":
    src_root = _find_pyproject_root(Path(CODE_DATASET_DIR))
    print("Copying OneVoice from Kaggle dataset:", src_root)
    for item in src_root.iterdir():
        dest = repo / item.name
        if item.is_dir():
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)
elif CODE_SOURCE == "github":
    if not REPO_URL:
        raise RuntimeError("CODE_SOURCE=github requires REPO_URL")
    clone_url = REPO_URL
    if USE_GITHUB_TOKEN:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            if token and clone_url.startswith("https://"):
                clone_url = "https://" + token + "@" + clone_url[len("https://") :]
        except Exception as exc:
            print("WARN: GITHUB_TOKEN:", exc)
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", clone_url, str(repo)],
        check=True,
    )
else:
    raise RuntimeError(f"unknown CODE_SOURCE: {CODE_SOURCE}")

os.chdir(repo)
if not (repo / "pyproject.toml").is_file():
    raise RuntimeError(f"missing pyproject.toml in {repo}")
print("cwd =", Path.cwd())


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dolphin,bench]"],
    check=True,
)
subprocess.run([sys.executable, "scripts/vendor_dolphin.py"], check=True)

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available — set Notebook Settings → Accelerator to GPU T4 and re-run."
    )
print("GPU:", torch.cuda.get_device_name(0))

from huggingface_hub import hf_hub_download
for fname in ("config.json", "model.safetensors"):
    path = hf_hub_download(MODEL_ID, filename=fname)
    print("HF", fname, "->", path)


In [ ]:
import json
import shutil
from pathlib import Path

src_tt = Path(DATASET_TT_DIR)
if not (src_tt / "mix.json").is_file():

    candidates = [
        Path("/kaggle/input"),
    ]
    found = None
    for root in candidates:
        for hit in root.rglob("mix.json"):
            if hit.parent.name == "tt" or (hit.parent / "corpus_mix_01").is_dir():
                found = hit.parent
                break
        if found:
            break
    if found is None:
        raise RuntimeError(
            f"DATASET_TT_DIR missing mix.json: {src_tt}. "
            "Attach the Kaggle Dataset that contains dolphin_tier_a/tt/."
        )
    print("Auto-detected tt dir:", found)
    src_tt = found

dst_tt = Path("/kaggle/working/dolphin_tier_a/tt")
if dst_tt.exists():
    shutil.rmtree(dst_tt)
shutil.copytree(src_tt, dst_tt)
print("Copied tt ->", dst_tt)

def rewrite_json(path: Path, tt_root: Path) -> None:
    data = json.loads(path.read_text(encoding="utf-8"))
    out = []
    for row in data:

        new = list(row)
        for i, cell in enumerate(row[:-1]):
            p = Path(str(cell))

            parts = p.parts
            if "corpus_mix_01" in parts or any(x.startswith("corpus_mix_") for x in parts):

                idx = next(j for j, x in enumerate(parts) if x.startswith("corpus_mix_"))
                rel = Path(*parts[idx:])
                new[i] = str((tt_root / rel).resolve())
            else:

                new[i] = str((tt_root / p.name).resolve())
            if not Path(new[i]).is_file():
                raise RuntimeError(f"missing after rewrite: {new[i]} (from {cell})")
        out.append(new)
    path.write_text(json.dumps(out, indent=2), encoding="utf-8")
    print("rewrote", path.name, "rows", len(out))

for name in ("mix.json", "s1.json", "s2.json"):
    rewrite_json(dst_tt / name, dst_tt)

TEST_DIR = str(dst_tt)
print("TEST_DIR =", TEST_DIR)
print("mixes:", sorted(p.name for p in dst_tt.glob("corpus_mix_*")))


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    "-m",
    "bench.bench_dolphin_offline",
    "--test-dir",
    TEST_DIR,
    "--segment-seconds",
    str(SEGMENT_SECONDS),
    "--max-clips",
    str(MAX_CLIPS),
    "--device",
    "cuda",
    "--fp16",
    "--dolphin-light",
    "--model-id",
    MODEL_ID,
    "--window-ms",
    str(WINDOW_MS),
    "--output-buffer-ms",
    str(OUTPUT_BUFFER_MS),
    "--output-dir",
    "/kaggle/working/runs",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"bench failed with exit {proc.returncode}")

runs = sorted(Path("/kaggle/working/runs").glob("*-dolphin_offline-*"))
if not runs:
    raise RuntimeError("no run directory written under /kaggle/working/runs")
run_dir = runs[-1]
metrics_path = run_dir / "metrics.json"
if not metrics_path.is_file():
    print("run files:", [p.name for p in run_dir.iterdir()])
    raise RuntimeError(f"metrics.json missing in {run_dir}")

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
dolphin = metrics.get("dolphin", {})
print("run_dir =", run_dir)
print("device =", dolphin.get("device"), "fp16 =", dolphin.get("fp16"), "light =", dolphin.get("dolphin_light"))
print("mean_infer_ms =", dolphin.get("mean_infer_ms"))
print("p95_infer_ms =", dolphin.get("p95_infer_ms"))
print("median SI-SNRi =", dolphin.get("median_si_snr_improvement_db"))
print("gate =", dolphin.get("gate_pass_count"), "/", dolphin.get("gate_total"))
RUN_DIR = run_dir
METRICS = metrics
DOLPHIN = dolphin


In [ ]:
import csv
from pathlib import Path

d = DOLPHIN
mean_ms = float(d["mean_infer_ms"])
p95_ms = float(d["p95_infer_ms"])
e2e_ms = float(d["buffered_e2e_ms"])
median_snri = float(d["median_si_snr_improvement_db"])
mean_snri = float(d["mean_si_snr_improvement_db"])
gate_pass = int(d["gate_pass_count"])
gate_total = int(d["gate_total"])

print("=== Latency (per 2s segment, CUDA FP16 dolphin_light) ===")
print(f"Mean infer: {mean_ms:.0f} ms")
print(f"P95 infer:  {p95_ms:.0f} ms")
print(
    f"Projected buffered E2E: {WINDOW_MS:.0f} (window) + {mean_ms:.0f} (infer) + "
    f"{OUTPUT_BUFFER_MS:.0f} (outbuf) = {e2e_ms:.0f} ms"
)
print()
print("=== Quality gate ===")
print(f"Median SI-SNRi: {median_snri:+.2f} dB")
print(f"Mean SI-SNRi:   {mean_snri:+.2f} dB")
print(f"Gate: {gate_pass}/{gate_total} targets >= 7 dB")

per_csv = RUN_DIR / "per_result.csv"
if per_csv.is_file():
    print()
    with open(per_csv, encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            snri = float(row["si_snr_improvement_db"])
            ims = float(row.get("infer_ms") or 0)
            mark = "PASS" if snri >= 7.0 else "FAIL"
            print(f"  {row['clip_id']}: SI-SNRi={snri:+.2f} dB infer={ims:.0f} ms [{mark}]")

viable = mean_ms < WINDOW_MS
VERDICT = (
    f"T4 per-segment = {mean_ms:.0f} ms -> buffered-live ~{e2e_ms:.0f} ms; "
    f"{'VIABLE' if viable else 'NOT VIABLE'} on an RTX Victus"
)
print()
print("VERDICT:", VERDICT)


In [ ]:
import subprocess
import sys
from pathlib import Path

export_root = Path(EXPORT_ROOT)
export_root.mkdir(parents=True, exist_ok=True)

for clip_id, target in EXPORT_TARGETS:
    out = export_root / f"{clip_id}_{target}"
    cmd = [
        sys.executable,
        "scripts/export_dolphin_separation.py",
        "--test-dir",
        TEST_DIR,
        "--clip-id",
        clip_id,
        "--target",
        target,
        "--segment-seconds",
        str(SEGMENT_SECONDS),
        "--device",
        "cuda",
        "--fp16",
        "--dolphin-light",
        "--out-dir",
        str(out),
    ]
    print("Exporting", clip_id, target, "->", out)
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f"export failed for {clip_id}/{target}")
    for p in sorted(out.glob("*")):
        print(" ", p.name, p.stat().st_size)

print("\nDownload from Kaggle Output:", export_root)
print("Files: 00_mixture_before.wav, 01_separated_after.wav, 02_clean_reference.wav")
print()
print("VERDICT:", VERDICT)
